# Feature engineering pipeline

### Importing the necessary libraries and session configuration

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    col, lit, when, coalesce,
    count, sum, avg, min, max,
    stddev, countDistinct, first, last,
    round, abs, sqrt, log,
    upper, trim, regexp_replace,
    to_date, datediff, current_date,
    months_between, year, month,
    date_format,
    row_number, lag, rank, ntile,
    percent_rank,
    broadcast, desc, asc
)
from pyspark.ml.feature import (
    VectorAssembler,
    StandardScaler,
    StringIndexer,
    OneHotEncoder,
    Imputer,
    Bucketizer
)
from pyspark.ml import Pipeline

In [2]:
spark = SparkSession.builder \
    .appName("BankingFeatureEngineering") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()
spark

- `config("spark.sql.shuffle.partitions", "8")` :  no of partitions
- `config("spark.driver.memory", "4g")` : memory allocated to the spark driver

### Loading all the tables

In [5]:
# Load all source tables
customers = spark.read.csv("../datasets/banking/customers.csv",
                header=True, inferSchema=True)

transactions = spark.read.csv("../datasets/transactions_windows.csv",
                header=True, inferSchema=True) \
    .withColumn("transaction_date",
        to_date(col("transaction_date"), "yyyy-MM-dd"))

loans = spark.read.csv("../datasets/banking/loans.csv",
                header=True, inferSchema=True) \
    .withColumn("disbursement_date",
        to_date(col("disbursement_date"), "yyyy-MM-dd"))

credit_scores = spark.read.csv("../datasets/banking/credit_scores.csv",
                header=True, inferSchema=True)

branches = spark.read.csv("../datasets/banking/branches.csv",
                header=True, inferSchema=True)

### Cache every table that will be used more than once, this prevents re-reading from disk on every join/aggregation

In [6]:
customers.cache()
transactions.cache()
loans.cache()
credit_scores.cache()
branches.cache()

# Triggering the cache by running a lightweight action
# Without this, caching is lazy, first real job still reads from disk
customers.count()
transactions.count()
loans.count()
credit_scores.count()
branches.count()

print("All tables loaded and cached.")
print(f"  Customers:    {customers.count()} rows")
print(f"  Transactions: {transactions.count()} rows")
print(f"  Loans:        {loans.count()} rows")
print(f"  Credit scores:{credit_scores.count()} rows")
print(f"  Branches:     {branches.count()} rows")

All tables loaded and cached.
  Customers:    8 rows
  Transactions: 20 rows
  Loans:        6 rows
  Credit scores:7 rows
  Branches:     7 rows


### Building the transaction based features

In [8]:
print("Building transaction features...")

# ──────────────────────── Window definitions ────────────────────────────────
w_cust_date = Window \
    .partitionBy("customer_id") \
    .orderBy("transaction_date")

w_cust_running = Window \
    .partitionBy("customer_id") \
    .orderBy("transaction_date") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

w_cust_full = Window \
    .partitionBy("customer_id") \
    .orderBy("transaction_date") \
    .rowsBetween(Window.unboundedPreceding,
                 Window.unboundedFollowing)

Building transaction features...


In [9]:
txn_enriched = transactions \
    .withColumn("txn_seq_num",
        row_number().over(w_cust_date)) \
    \
    .withColumn("prev_amount",
        lag("transaction_amount", 1, 0.0).over(w_cust_date)) \
    \
    .withColumn("prev_date",
        lag("transaction_date", 1).over(w_cust_date)) \
    \
    .withColumn("days_since_prev_txn",
        when(col("prev_date").isNotNull(),
             datediff(col("transaction_date"),
                      col("prev_date")))
        .otherwise(lit(None))) \
    \
    .withColumn("running_avg",
        avg("transaction_amount").over(w_cust_running)) \
    \
    .withColumn("running_std",
        stddev("transaction_amount").over(w_cust_running)) \
    \
    .withColumn("is_spike",
        when(
            (col("prev_amount") > 0) &
            (col("transaction_amount") > col("prev_amount") * 3),
            1
        ).otherwise(0)) \
    \
    .withColumn("is_anomaly",
        when(
            col("running_std").isNotNull() &
            (col("transaction_amount") >
             col("running_avg") + 2 * col("running_std")),
            1
        ).otherwise(0)) \
    \
    .withColumn("is_debit",
        when(col("transaction_type") == "Debit", 1)
        .otherwise(0)) \
    \
    .withColumn("is_credit",
        when(col("transaction_type") == "Credit", 1)
        .otherwise(0))

### Transaction level --> customer level

In [11]:
txn_features = txn_enriched.groupBy("customer_id").agg(

    # Volume features
    count("*").alias("f_txn_count"),
    round(sum("transaction_amount"), 2)
        .alias("f_total_txn_volume"),
    round(avg("transaction_amount"), 2)
        .alias("f_avg_txn_amount"),
    round(min("transaction_amount"), 2)
        .alias("f_min_txn_amount"),
    round(max("transaction_amount"), 2)
        .alias("f_max_txn_amount"),
    round(stddev("transaction_amount"), 2)
        .alias("f_std_txn_amount"),

    # Debit / Credit split
    sum("is_debit").alias("f_debit_count"),
    sum("is_credit").alias("f_credit_count"),
    round(sum(
        when(col("transaction_type") == "Debit",
             col("transaction_amount")).otherwise(0)
        ), 2).alias("f_total_debit_amount"),
    round(sum(
        when(col("transaction_type") == "Credit",
             col("transaction_amount")).otherwise(0)
        ), 2).alias("f_total_credit_amount"),

    # Behavioural flags
    sum("is_spike").alias("f_spike_count"),
    sum("is_anomaly").alias("f_anomaly_count"),

    # Recency features
    max("transaction_date").alias("f_last_txn_date"),
    min("transaction_date").alias("f_first_txn_date"),
    round(avg("days_since_prev_txn"), 1)
        .alias("f_avg_days_between_txns"),

    # Balance features
    round(avg("balance"), 2).alias("f_avg_balance"),
    round(min("balance"), 2).alias("f_min_balance"),
    round(max("balance"), 2).alias("f_max_balance"),

    # Recency (days since last transaction)
    max("transaction_date").alias("_last_date")

).withColumn("f_days_since_last_txn",
    datediff(current_date(), col("_last_date"))
).drop("_last_date")

In [12]:
txn_features.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- f_txn_count: long (nullable = false)
 |-- f_total_txn_volume: double (nullable = true)
 |-- f_avg_txn_amount: double (nullable = true)
 |-- f_min_txn_amount: double (nullable = true)
 |-- f_max_txn_amount: double (nullable = true)
 |-- f_std_txn_amount: double (nullable = true)
 |-- f_debit_count: long (nullable = true)
 |-- f_credit_count: long (nullable = true)
 |-- f_total_debit_amount: double (nullable = true)
 |-- f_total_credit_amount: double (nullable = true)
 |-- f_spike_count: long (nullable = true)
 |-- f_anomaly_count: long (nullable = true)
 |-- f_last_txn_date: date (nullable = true)
 |-- f_first_txn_date: date (nullable = true)
 |-- f_avg_days_between_txns: double (nullable = true)
 |-- f_avg_balance: double (nullable = true)
 |-- f_min_balance: double (nullable = true)
 |-- f_max_balance: double (nullable = true)
 |-- f_days_since_last_txn: integer (nullable = true)



### Ratio features

In [14]:
txn_features = txn_features \
    .withColumn("f_debit_ratio",
        round(
            col("f_debit_count") /
            when(col("f_txn_count") > 0, col("f_txn_count"))
            .otherwise(lit(1)),
            4
        )) \
    \
    .withColumn("f_credit_ratio",
        round(1 - col("f_debit_ratio"), 4)) \
    \
    .withColumn("f_avg_balance_to_avg_txn",
        round(
            when(col("f_avg_txn_amount") > 0,
                 col("f_avg_balance") / col("f_avg_txn_amount"))
            .otherwise(lit(None)),
            4
        )) \
    \
    .withColumn("f_spike_rate",
        round(
            col("f_spike_count") /
            when(col("f_txn_count") > 0, col("f_txn_count"))
            .otherwise(lit(1)),
            4
        )) \
    \
    .withColumn("f_customer_tenure_months",
        round(
            months_between(current_date(), col("f_first_txn_date")),
            1
        ))

print(f"Transaction features: {txn_features.count()} customers")
txn_features.printSchema()

Transaction features: 7 customers
root
 |-- customer_id: string (nullable = true)
 |-- f_txn_count: long (nullable = false)
 |-- f_total_txn_volume: double (nullable = true)
 |-- f_avg_txn_amount: double (nullable = true)
 |-- f_min_txn_amount: double (nullable = true)
 |-- f_max_txn_amount: double (nullable = true)
 |-- f_std_txn_amount: double (nullable = true)
 |-- f_debit_count: long (nullable = true)
 |-- f_credit_count: long (nullable = true)
 |-- f_total_debit_amount: double (nullable = true)
 |-- f_total_credit_amount: double (nullable = true)
 |-- f_spike_count: long (nullable = true)
 |-- f_anomaly_count: long (nullable = true)
 |-- f_last_txn_date: date (nullable = true)
 |-- f_first_txn_date: date (nullable = true)
 |-- f_avg_days_between_txns: double (nullable = true)
 |-- f_avg_balance: double (nullable = true)
 |-- f_min_balance: double (nullable = true)
 |-- f_max_balance: double (nullable = true)
 |-- f_days_since_last_txn: integer (nullable = true)
 |-- f_debit_ratio:

### Loan based features

In [16]:
print("Building loan features...")

loan_features = loans.groupBy("customer_id").agg(

    count("*").alias("f_total_loans"),

    # Loan type breakdown
    sum(when(col("loan_type") == "Home Loan", 1)
        .otherwise(0)).alias("f_home_loans"),
    sum(when(col("loan_type") == "Personal Loan", 1)
        .otherwise(0)).alias("f_personal_loans"),
    sum(when(col("loan_type") == "Car Loan", 1)
        .otherwise(0)).alias("f_car_loans"),

    # Loan status breakdown
    sum(when(col("status") == "Active", 1)
        .otherwise(0)).alias("f_active_loans"),
    sum(when(col("status") == "Defaulted", 1)
        .otherwise(0)).alias("f_defaulted_loans"),
    sum(when(col("status") == "Closed", 1)
        .otherwise(0)).alias("f_closed_loans"),

    # Loan amounts
    round(sum("loan_amount"), 2)
        .alias("f_total_loan_amount"),
    round(avg("loan_amount"), 2)
        .alias("f_avg_loan_amount"),
    round(max("loan_amount"), 2)
        .alias("f_max_loan_amount"),
    round(avg("interest_rate"), 2)
        .alias("f_avg_interest_rate"),

    # Loan tenure — months since oldest loan
    round(months_between(
        current_date(),
        min("disbursement_date")), 1
    ).alias("f_months_since_first_loan")

) \
.withColumn("f_has_active_loan",
    when(col("f_active_loans") > 0, 1).otherwise(0)) \
.withColumn("f_has_defaulted",
    when(col("f_defaulted_loans") > 0, 1).otherwise(0)) \
.withColumn("f_default_rate",
    round(
        col("f_defaulted_loans") / col("f_total_loans"),
        4
    ))

print(f"Loan features: {loan_features.count()} customers")

Building loan features...
Loan features: 6 customers


### Credit score features

In [18]:
print("Building credit score features...")

credit_features = credit_scores.select(
    "customer_id",
    col("credit_score").alias("f_credit_score"),
    col("bureau_name").alias("f_bureau_name"),

    # Risk band — derived from credit score
    when(col("credit_score") >= 750, "Excellent")
    .when(col("credit_score") >= 700, "Good")
    .when(col("credit_score") >= 650, "Fair")
    .when(col("credit_score") >= 600, "Poor")
    .otherwise("No Score")
    .alias("f_risk_band"),

    # Binary risk flag for ML
    when(col("credit_score") >= 700, 0)
    .otherwise(1)
    .alias("f_is_high_risk")
)

print(f"Credit features: {credit_features.count()} customers")

Building credit score features...
Credit features: 7 customers


### Customer features

In [20]:
print("Building customer base features...")

customer_features = customers.select(
    "customer_id",
    col("name").alias("customer_name"),
    col("age").alias("f_age"),
    col("city").alias("f_city"),
    col("segment").alias("f_segment"),
    col("annual_income").alias("f_annual_income"),

    # Age bands
    when(col("age") < 30, "Young")
    .when(col("age") < 45, "Mid")
    .when(col("age") < 60, "Senior")
    .otherwise("Elder")
    .alias("f_age_band"),

    # Income bands
    when(col("annual_income") >= 1000000, "High")
    .when(col("annual_income") >= 500000, "Medium")
    .otherwise("Low")
    .alias("f_income_band"),

    # Customer tenure from join date
    round(
        months_between(current_date(),
            to_date(col("join_date"), "yyyy-MM-dd")),
        1
    ).alias("f_tenure_months")
)

print(f"Customer base features: {customer_features.count()} customers")

Building customer base features...
Customer base features: 8 customers


### Combining the features in a single table

In [22]:
print("Assembling master feature table...")

feature_table = customer_features \
    .join(txn_features,
          on="customer_id", how="left") \
    .join(loan_features,
          on="customer_id", how="left") \
    .join(broadcast(credit_features),
          on="customer_id", how="left")

Assembling master feature table...


### Handling the null values after join 

In [24]:
feature_table = feature_table.na.fill({
    # Transaction features — 0 for customers with no transactions
    "f_txn_count":              0,
    "f_total_txn_volume":       0.0,
    "f_avg_txn_amount":         0.0,
    "f_min_txn_amount":         0.0,
    "f_max_txn_amount":         0.0,
    "f_debit_count":            0,
    "f_credit_count":           0,
    "f_total_debit_amount":     0.0,
    "f_total_credit_amount":    0.0,
    "f_spike_count":            0,
    "f_anomaly_count":          0,
    "f_debit_ratio":            0.0,
    "f_credit_ratio":           0.0,
    "f_spike_rate":             0.0,

    # Loan features — 0 for customers with no loans
    "f_total_loans":            0,
    "f_active_loans":           0,
    "f_defaulted_loans":        0,
    "f_closed_loans":           0,
    "f_total_loan_amount":      0.0,
    "f_has_active_loan":        0,
    "f_has_defaulted":          0,
    "f_default_rate":           0.0,

    # Credit — default for customers not in credit bureau
    "f_risk_band":              "No Score",
    "f_is_high_risk":           1
})

print(f"\nMaster feature table: {feature_table.count()} customers")
print(f"Total features: {len(feature_table.columns)} columns")
feature_table.printSchema()
feature_table.show(5, truncate=False)


Master feature table: 8 customers
Total features: 52 columns
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- f_age: integer (nullable = true)
 |-- f_city: string (nullable = true)
 |-- f_segment: string (nullable = true)
 |-- f_annual_income: integer (nullable = true)
 |-- f_age_band: string (nullable = false)
 |-- f_income_band: string (nullable = false)
 |-- f_tenure_months: double (nullable = true)
 |-- f_txn_count: long (nullable = false)
 |-- f_total_txn_volume: double (nullable = false)
 |-- f_avg_txn_amount: double (nullable = false)
 |-- f_min_txn_amount: double (nullable = false)
 |-- f_max_txn_amount: double (nullable = false)
 |-- f_std_txn_amount: double (nullable = true)
 |-- f_debit_count: long (nullable = false)
 |-- f_credit_count: long (nullable = false)
 |-- f_total_debit_amount: double (nullable = false)
 |-- f_total_credit_amount: double (nullable = false)
 |-- f_spike_count: long (nullable = false)
 |-- f_anomaly_c

### Preparing for the ML Pipeline

#### Handling missing nulls using imputer

In [27]:
# Impute any remaining numeric nulls with median for customers who had transactions but some features still null

numeric_features_to_impute = [
    "f_std_txn_amount",
    "f_avg_days_between_txns",
    "f_avg_balance_to_avg_txn",
    "f_credit_score",
    "f_avg_interest_rate",
    "f_months_since_first_loan",
    "f_customer_tenure_months",
    "f_days_since_last_txn"
]

imputer = Imputer(
    inputCols=numeric_features_to_impute,
    outputCols=numeric_features_to_impute 
).setStrategy("median")

feature_table = imputer.fit(feature_table) \
                       .transform(feature_table)

### Encoding categorical features

In [29]:
# StringIndexer converts string categories to numeric indices
# Encode f_segment: "Basic" → 0, "Standard" → 1, "Premium" → 2
segment_indexer = StringIndexer(
    inputCol="f_segment",
    outputCol="f_segment_idx",
    handleInvalid="keep"  # keep unseen categories as a new index
)

- `StringIndexer` — converts a string column to a numeric index column. Most frequent value gets index 0. Second most frequent gets 1, and so on.
- `handleInvalid="keep"` — if a value appears in test data that wasn't in training data (new city, new segment), don't throw an error — assign it to a special "unknown" index.

In [30]:
segment_encoder = OneHotEncoder(
    inputCols=["f_segment_idx"],
    outputCols=["f_segment_vec"]
)

- OneHotEncoder — converts the index into a sparse binary vector.

- "Basic" → index 0 → [1, 0, 0]
- "Standard" → index 1 → [0, 1, 0]
- "Premium" → index 2 → [0, 0, 1]

one hot encoder is used after the indexing so that the model doesn't assume any order in the index and give some index more priority than the other

### Encoding the risk bands

In [31]:
risk_indexer = StringIndexer(
    inputCol="f_risk_band",
    outputCol="f_risk_band_idx",
    handleInvalid="keep"
)

risk_encoder = OneHotEncoder(
    inputCols=["f_risk_band_idx"],
    outputCols=["f_risk_band_vec"]
)

### Bucketizing the continuous feature like age

In [33]:
age_bucketizer = Bucketizer(
    splits=[0, 25, 35, 50, 65, float("inf")],
    inputCol="f_age",
    outputCol="f_age_bucket"
)

- `Bucketizer` — bins a continuous column into discrete buckets based on boundaries you define.

- splits=[0, 25, 35, 50, 65, inf] creates 5 buckets:

Bucket 0: age 0–25

Bucket 1: age 25–35

Bucket 2: age 35–50

Bucket 3: age 50–65

Bucket 4: age 65+

### Packing all features in a single vector as the pyspark model expects it in this single vectorized format

In [35]:
numeric_cols = [
    "f_age",
    "f_annual_income",
    "f_tenure_months",
    "f_txn_count",
    "f_total_txn_volume",
    "f_avg_txn_amount",
    "f_max_txn_amount",
    "f_std_txn_amount",
    "f_debit_ratio",
    "f_spike_count",
    "f_anomaly_count",
    "f_spike_rate",
    "f_avg_days_between_txns",
    "f_days_since_last_txn",
    "f_avg_balance",
    "f_min_balance",
    "f_max_balance",
    "f_avg_balance_to_avg_txn",
    "f_total_loans",
    "f_active_loans",
    "f_defaulted_loans",
    "f_total_loan_amount",
    "f_avg_interest_rate",
    "f_has_active_loan",
    "f_has_defaulted",
    "f_default_rate",
    "f_credit_score",
    "f_is_high_risk",
    "f_age_bucket"
]

# Encoded categorical vectors
encoded_cols = [
    "f_segment_vec",
    "f_risk_band_vec"
]

assembler = VectorAssembler(
    inputCols=numeric_cols + encoded_cols,
    outputCol="features",
    handleInvalid="keep"
)

### Scaling the data

In [37]:
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withMean=True,   
    withStd=True 
)

### Building the pipeline for consistency

In [39]:
# Pipeline chains all steps in order — fit once, transform consistently
ml_pipeline = Pipeline(stages=[
    segment_indexer,    # String → index
    risk_indexer,       # String → index
    segment_encoder,    # index → one-hot vector
    risk_encoder,       # index → one-hot vector
    age_bucketizer,     # continuous → bucket
    assembler,          # all columns → one vector
    scaler              # vector → scaled vector
])

In [40]:
# Fit the pipeline on the feature table
print("Fitting ML pipeline...")
pipeline_model = ml_pipeline.fit(feature_table)

print("Transforming data...")
ml_ready = pipeline_model.transform(feature_table)


ml_ready.select(
    "customer_id",
    "customer_name",
    "f_credit_score",
    "f_txn_count",
    "f_has_defaulted",
    "f_risk_band",
    "features",
    "scaled_features"
).show(5, truncate=False)

Fitting ML pipeline...
Transforming data...
+-----------+-------------+--------------+-----------+---------------+-----------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|customer_id|customer_name|f_credit_score|f_txn_cou